Подготовка дпнных

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from statsmodels.stats.proportion import proportions_ztest, confint_proportions_2indep

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("Set2")
plt.rcParams['figure.figsize'] = (12, 6)


In [2]:
df = pd.read_csv('ab_data.csv')
countries  = pd.read_csv('countries.csv')
df = pd.merge(df,countries, on='user_id', how='left')

EDA-анализ датасета

In [3]:
final_group = ( 
    df.groupby(['group','landing_page'], as_index=False)
    .agg(users=('user_id','count'))
    )

In [4]:
final_group 

,group,landing_page,users
0,control,new_page,1928
1,control,old_page,145274
2,treatment,new_page,145315
3,treatment,old_page,1965


In [5]:
len(df)

294482

Смотрим на таблицу mapping_check и делаем вывод о томчто у нас присутствуют данные которые не подходят а именно (control - new) и (treatmeat - old),потому что нас интересуют две группы, control - где старая страница и treatmeat - новая страница , остальное удаляем из наших данных . Всего строк 294482

In [6]:
df = df[((df['group'] == 'control') & (df['landing_page'] == 'old_page'))|
        ((df['group'] == 'treatment') & (df['landing_page'] == 'new_page'))]

In [7]:
len(df)

290589

строк после удаления 290589

In [8]:
df.describe()

,user_id,converted
count,290589.000000,290589.000000
mean,788004.037913,0.119595
std,91224.393160,0.324488
min,630000.000000,0.000000
25%,709035.000000,0.000000
50%,787992.000000,0.000000
75%,866955.000000,0.000000
max,945999.000000,1.000000


In [9]:
df.shape

(290589, 6)

In [10]:
df.isna().count()

user_id         290589
timestamp       290589
group           290589
landing_page    290589
converted       290589
country         290589
dtype: int64

In [11]:
int(df.duplicated().sum())
df[df.duplicated(keep=False)]

,user_id,timestamp,group,landing_page,converted,country
250001,759899,07:36.1,treatment,new_page,0,US
250002,759899,07:36.1,treatment,new_page,0,US
294479,759899,20:29.0,treatment,new_page,0,US
294480,759899,20:29.0,treatment,new_page,0,US


In [12]:
df['user_id'].nunique()

290585

In [13]:
df = df.drop_duplicates(subset=['user_id'], keep='last')

In [14]:
df

,user_id,timestamp,group,landing_page,converted,country
0,851104,11:48.6,control,old_page,0,US
1,804228,01:45.2,control,old_page,0,US
2,661590,55:06.2,treatment,new_page,0,US
3,853541,28:03.1,treatment,new_page,0,US
4,864975,52:26.2,control,old_page,1,US
...,...,...,...,...,...,...
294476,734608,45:03.4,control,old_page,0,US
294477,697314,20:29.0,control,old_page,0,US
294478,715931,40:24.5,treatment,new_page,0,UK
294480,759899,20:29.0,treatment,new_page,0,US


Смотрим на сбаланисированность групп

In [15]:
group_dist = df.groupby('group',as_index=False).agg(count_r = ('user_id', 'count'))

In [16]:
group_dist['persent'] = round((group_dist['count_r'] / 290585)*100,2)

In [17]:
group_dist

,group,count_r,persent
0,control,145274,49.99
1,treatment,145311,50.01


In [18]:
diff = abs(group_dist.loc[0, 'persent'] - group_dist.loc[1, 'persent'])

In [19]:
round(float(diff),2)

0.02

Вывод : группы сбалансированы ,потому что разница между ними меньше 0,02%

Проверяем данные

In [20]:
summary_df = (
    df.groupby('group',as_index=False)
    .agg(
        users = ('user_id', 'count'),
        converted_users = ('converted', 'sum')
    )
    )

In [21]:
summary_df

,group,users,converted_users
0,control,145274,17489
1,treatment,145311,17264


In [22]:
# Рассчитываем коэффициент конверсии
summary_df['CR'] = round(summary_df['converted_users'] / summary_df['users']*100,2)

# Определяем группы
conversions = [summary_df[summary_df['group']=='control']['converted_users'].values[0],summary_df[summary_df['group']=='treatment']['converted_users'].values[0]]
total = [summary_df[summary_df['group']=='control']['users'].values[0],summary_df[summary_df['group']=='treatment']['users'].values[0]]

# Пропорциональный z-тест
zstat,p_val = proportions_ztest(count=conversions, nobs=total)

In [23]:
float(p_val)

0.18965258971881804

Делаем вывод по p-value,наше значение (0.18965) > допустимого порога 0.05 ,поэтому мы не можем отвергнуть гипотезу H0 hgfdfgh